## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes duckduckgo-search trafilatura requests


## 2. Load the open-source LLM

Qwen2.5-7B-Instruct in 4-bit fits in ~6GB VRAM. If you're on a smaller GPU or want faster iterations
while testing, swap `MODEL_NAME` for `"microsoft/Phi-3-mini-4k-instruct"` (3.8B, much faster, slightly
weaker reasoning).


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # or "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

def llm_generate(prompt: str, system: str = "You are a careful, precise research assistant.",
                  max_new_tokens: int = 700, temperature: float = 0.3) -> str:
    """Run one chat-style generation through the local model."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Model loaded.")


## 3. Web search + page extraction tools

In [ ]:
import re
import json
import requests
import trafilatura
from duckduckgo_search import DDGS

def search_web(query: str, max_results: int = 5):
    """Live DuckDuckGo search. Returns list of {title, href, snippet}."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        return [{"title": r.get("title", ""), "href": r.get("href", ""),
                  "snippet": r.get("body", "")} for r in results]
    except Exception as e:
        print(f"  [search error for '{query}']: {e}")
        return []

def fetch_page_text(url: str, max_chars: int = 4000) -> str:
    """Download and extract clean article text from a URL."""
    try:
        downloaded = trafilatura.fetch_url(url, timeout=10)
        if not downloaded:
            return ""
        text = trafilatura.extract(downloaded) or ""
        return text[:max_chars]
    except Exception:
        return ""

def extract_json(text: str):
    """Best-effort extraction of a JSON object/array from an LLM response."""
    match = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


## 4. The agent

Each stage is its own function so you can inspect, tweak, or swap any piece independently.


In [ ]:
class ResearchAgent:
    def __init__(self, max_iterations: int = 3, results_per_query: int = 4, relevance_threshold: int = 3):
        self.max_iterations = max_iterations
        self.results_per_query = results_per_query
        self.relevance_threshold = relevance_threshold

    # --- 1. Plan ---------------------------------------------------
    def plan_queries(self, question: str, prior_context: str = "") -> list:
        prompt = f"""Question to research: \"{question}\"

{f"So far we have found: {prior_context}" if prior_context else ""}

Generate 2-4 focused web search queries that would help answer this question thoroughly.
Respond with ONLY a JSON array of strings, nothing else. Example: ["query one", "query two"]"""
        raw = llm_generate(prompt, temperature=0.4)
        queries = extract_json(raw)
        if not isinstance(queries, list) or not queries:
            queries = [question]
        return queries[:4]

    # --- 2. Search + fetch ------------------------------------------
    def gather_sources(self, queries: list) -> list:
        sources = []
        for q in queries:
            print(f"  Searching: {q}")
            for r in search_web(q, max_results=self.results_per_query):
                text = fetch_page_text(r["href"])
                if text:
                    sources.append({"query": q, "title": r["title"], "url": r["href"], "text": text})
        return sources

    # --- 3. Validate --------------------------------------------------
    def validate_sources(self, question: str, sources: list) -> list:
        validated = []
        for s in sources:
            prompt = f"""Research question: \"{question}\"

Source title: {s['title']}
Source excerpt: {s['text'][:1200]}

Rate this source's relevance and reliability for answering the question, 1 (useless) to 5 (excellent).
Respond with ONLY a JSON object: {{"score": <int>, "reason": "<one short sentence>"}}"""
            raw = llm_generate(prompt, temperature=0.1, max_new_tokens=150)
            verdict = extract_json(raw) or {}
            score = verdict.get("score", 0)
            if isinstance(score, (int, float)) and score >= self.relevance_threshold:
                s["relevance_score"] = score
                s["relevance_reason"] = verdict.get("reason", "")
                validated.append(s)
        return validated

    # --- 4. Synthesize --------------------------------------------------
    def synthesize(self, question: str, sources: list) -> str:
        if not sources:
            return "No sufficiently relevant sources were found."
        evidence = "\n\n".join(
            f"[{i+1}] {s['title']} ({s['url']})\n{s['text'][:1000]}"
            for i, s in enumerate(sources)
        )
        prompt = f"""Question: \"{question}\"

Evidence from sources:
{evidence}

Write a clear, well-organized answer to the question using ONLY this evidence.
Cite sources inline using [1], [2], etc. matching the numbers above.
If the evidence is incomplete or conflicting, say so explicitly."""
        return llm_generate(prompt, temperature=0.3, max_new_tokens=800)

    # --- 5. Reflect -------------------------------------------------
    def reflect(self, question: str, answer: str) -> list:
        prompt = f"""Question: \"{question}\"

Current draft answer:
{answer}

Does this answer fully and accurately address the question? If yes, respond with exactly: DONE
If not, respond with ONLY a JSON array of 1-3 follow-up search queries that would fill the gaps."""
        raw = llm_generate(prompt, temperature=0.3, max_new_tokens=200)
        if "DONE" in raw.upper() and extract_json(raw) is None:
            return []
        queries = extract_json(raw)
        return queries if isinstance(queries, list) else []

    # --- Orchestration -------------------------------------------------
    def run(self, question: str) -> dict:
        print(f"Question: {question}\n")
        all_sources, answer, iteration = [], "", 0
        prior_context = ""

        while iteration < self.max_iterations:
            iteration += 1
            print(f"--- Iteration {iteration} ---")

            queries = self.plan_queries(question, prior_context)
            print(f"Planned queries: {queries}")

            raw_sources = self.gather_sources(queries)
            print(f"Fetched {len(raw_sources)} pages")

            good_sources = self.validate_sources(question, raw_sources)
            print(f"Validated {len(good_sources)} as relevant (score >= {self.relevance_threshold})")

            for s in good_sources:
                if not any(s['url'] == existing['url'] for existing in all_sources):
                    all_sources.append(s)

            answer = self.synthesize(question, all_sources)
            prior_context = answer[:500]

            follow_ups = self.reflect(question, answer)
            if not follow_ups:
                print("Agent judged the answer sufficient. Stopping.\n")
                break
            print(f"Agent wants more info, follow-up queries: {follow_ups}\n")

        return {"question": question, "answer": answer, "sources": all_sources, "iterations": iteration}


## 5. Run it

In [ ]:
agent = ResearchAgent(max_iterations=3, results_per_query=4, relevance_threshold=3)

result = agent.run("What are the current best practices for reducing hallucinations in retrieval-augmented generation systems?")

print("=" * 80)
print("FINAL ANSWER")
print("=" * 80)
print(result["answer"])
print("\n" + "=" * 80)
print(f"SOURCES USED ({len(result['sources'])}), across {result['iterations']} iteration(s)")
print("=" * 80)
for i, s in enumerate(result["sources"]):
    print(f"[{i+1}] {s['title']} — {s['url']}  (relevance: {s.get('relevance_score')})")
